# 📊 Exploratory Data Analysis — HedgeForge

**Project:** HedgeForge — Portfolio Optimization & Risk Modeling
**Phase:** 2 — Data Engineering & Validation
**Author:** Brice Nelson
**Date:** {{CURRENT_DATE}}

---

## 🎯 Objective

This notebook performs **Exploratory Data Analysis (EDA)** on the market and portfolio datasets that will power HedgeForge’s optimization and risk models.
The goal is to ensure **data integrity**, understand **distributional characteristics**, and surface **risk-relevant insights** prior to modeling.

---

## 🧩 Key Analysis Areas

| Focus Area | Purpose |
|-------------|----------|
| **Data Quality & Validation** | Identify missing values, duplicates, and structural inconsistencies. |
| **Statistical Overview** | Summarize returns, volatility, and correlations across assets. |
| **Distribution Analysis** | Evaluate skewness, kurtosis, and heavy-tail behavior in returns. |
| **Correlation & Covariance** | Explore inter-asset dependencies for risk modeling. |
| **Stationarity Checks** | Verify statistical properties required for Monte Carlo and VaR analysis. |
| **Outlier Detection** | Identify anomalous behavior indicative of stress scenarios. |

---

## 🧠 Insights to Extract

- Are return distributions **normal, skewed, or heavy-tailed**?
- Which assets contribute most to **systematic risk**?
- Do correlations **tighten under stress** (volatility clustering)?
- Are there any **data reliability issues** that could bias VaR/CVaR estimates?

---

## 🚀 Next Steps

1. Clean and validate raw data (`validate_data()`).
2. Compute log returns (`compute_log_returns()`).
3. Generate covariance and correlation matrices.
4. Visualize volatility and dependency structures.
5. Export processed data to `/data/processed/` for optimization modules.

---

> 🧾 *This notebook bridges the raw data ingestion phase and the optimization engine.
It documents all assumptions, validation rules, and preprocessing decisions applied to the HedgeForge dataset.*


In [1]:
import numpy as np
import pandas as pd
import matplotlib as plt
import seaborn as sns

# imports for processing data prior to creating dataframes
from pathlib import Path
from scripts.clean_currency_csvs import clean_csv


In [2]:
# data is not importing.  Determine the issue with below code
import csv
file_path = "../data/raw/portfolio_acc001_taxable.csv"

with open(file_path, "r") as f:
    reader = csv.reader(f)
    for i, row in enumerate(reader, start=1):
        if len(row) != 17:
            print(f"Line {i}: {len(row)} fields → {row}")


Line 1: 16 fields → ['Account Number', 'Tax Lot ID', 'Ticker', 'CUSIP', 'Security Name', 'Asset Type', 'Sector', 'Purchase Date', 'Quantity', 'Cost Basis', 'Total Cost', 'Trade Fees', 'Current Price', 'Market Value', 'Unrealized Gain/Loss', 'Currency']
Line 3: 19 fields → ['ACC-001', 'LOT-002', 'AAPL', '037833100', 'Apple Inc.', 'Equity', 'Technology', '2019-11-01', '500', '$65.00', '$32', '500.00', '$7.95', '$198.00', '$99', '000.00', '$66', '500.00', 'USD']
Line 4: 18 fields → ['ACC-001', 'LOT-003', 'NEM', '651639106', 'Newmont Corporation', 'Equity', 'Commodities', '2018-10-10', '100', '$33.00', '$3', '300.00', '', '42', '$4', '200.00', '$900.00', 'USD']
Line 5: 19 fields → ['ACC-001', 'LOT-004', 'MMM', '88579Y101', '3M Co.', 'Equity', 'Industrials', '2019-12-12', '40', '$170', '$6', '800.00', '', '$105', '$4', '200.00', '-$2', '600.00', 'USD']
Line 6: 19 fields → ['ACC-001', 'LOT-005', 'SCCO', '84265V105', 'Southern Copper Corp', 'Equity', 'Commodities', '2019-03-20', '150', '$35.0

## Data Diagnosis & Solution
- data contains dollar signs and commas in numeric values and not wrapped with quotes.
- run script to remove objects
- reusable function saved in scripts folder
- save as clean file in raw folder

In [13]:
# Clean and reload data
cleaned_taxable = clean_csv(Path("../data/raw/portfolio_acc001_taxable.csv"))
cleaned_ira = clean_csv(Path("../data/raw/portfolio_acc002_ira.csv"))


🔍 Cleaning portfolio_acc001_taxable.csv ...
✅ Cleaned file saved to ../data/raw/portfolio_acc001_taxable_clean.csv
🔍 Cleaning portfolio_acc002_ira.csv ...
✅ Cleaned file saved to ../data/raw/portfolio_acc002_ira_clean.csv


In [14]:
# Verify files have been cleaned
import csv

with open(cleaned_taxable) as f:
    reader = csv.reader(f)
    for i, row in enumerate(reader, start=1):
        if len(row) != 17:
            print(f"⚠️ Line {i}: {len(row)} fields")


⚠️ Line 1: 16 fields
⚠️ Line 3: 19 fields
⚠️ Line 4: 18 fields
⚠️ Line 5: 19 fields
⚠️ Line 6: 19 fields
⚠️ Line 7: 21 fields
⚠️ Line 8: 18 fields
⚠️ Line 9: 19 fields
⚠️ Line 10: 19 fields
⚠️ Line 11: 19 fields
⚠️ Line 12: 19 fields
⚠️ Line 13: 19 fields
⚠️ Line 14: 19 fields
⚠️ Line 15: 19 fields
⚠️ Line 16: 19 fields
⚠️ Line 17: 19 fields
⚠️ Line 18: 20 fields
⚠️ Line 19: 19 fields
⚠️ Line 20: 19 fields
⚠️ Line 21: 19 fields
⚠️ Line 22: 18 fields
⚠️ Line 23: 19 fields


In [5]:
# create dataframes
acc01_tax_df = pd.read_csv("../data/raw/portfolio_acc001_taxable_clean.csv")
acc02_ira_df = pd.read_csv("../data/raw/portfolio_acc002_ira_clean.csv")

ParserError: Error tokenizing data. C error: Expected 17 fields in line 3, saw 19


In [ ]:
print(f'Taxable Acct: {acc01_tax.shape}')